# 教學網站 demo 產生器（在 Colab GPU 上標籤）

自動抓 3 支 CC0/公網域原片 → YOLO+ByteTrack 標籤 → 存 before/after 到 Drive `yolo-course/demos/`。
標籤在 Colab 真實跑（非本地）。執行階段請先選 **T4 GPU**，然後 Run All。

In [7]:
# 1) 確認加速器（demo 產生屬輕量推論，沒 GPU 也能用 CPU 跑，只是慢一點）
import torch
if torch.cuda.is_available():
    print('使用 GPU:', torch.cuda.get_device_name(0))
else:
    print('沒有 GPU → 改用 CPU（短片可接受，約多花幾分鐘）')

沒有 GPU → 改用 CPU（短片可接受，約多花幾分鐘）


In [8]:
# 2) 安裝
!pip -q install ultralytics
import ultralytics; print('ultralytics', ultralytics.__version__)

ultralytics 8.4.78


In [9]:
# 3) 掛 Drive
from google.colab import drive
drive.mount('/content/drive')
import os
WORK='/content/drive/MyDrive/yolo-course'
DEMOS=f'{WORK}/demos'
os.makedirs(DEMOS, exist_ok=True)
print('輸出到：', DEMOS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
輸出到： /content/drive/MyDrive/yolo-course/demos


## 場景設定（CC0 / 公網域，可公開重製）
| name | 來源 | 授權 |
|---|---|---|
| pedestrians | OpenCV vtest.avi | Apache-2.0 |
| street_lagos | Wikimedia: Oshodi, Lagos (Ready Street) | CC BY-SA 4.0 |
| leeds_1888 | Wikimedia: Leeds Bridge 1888 | Public Domain |

In [10]:
SCENES=[
  {'name':'pedestrians','title':'行人穿越','sec':15,
   'url':'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi'},
  {'name':'street_lagos','title':'繁忙街景（人+車+機車）','sec':15,
   'url':'https://upload.wikimedia.org/wikipedia/commons/2/2d/Video_of_the_Activities_in_Oshodi%2C_Lagos%2C_Nigeria.webm'},
  {'name':'leeds_1888','title':'1888 歷史影片','sec':15,
   'url':'https://upload.wikimedia.org/wikipedia/commons/2/25/Traffic_Crossing_Leeds_Bridge_film.ogv'},
]
H, FPS, CONF = 360, 15, 0.30

In [11]:
# 4) 下載 → 裁切 → YOLO 標籤 → 存 before/after（H264）到 Drive
import subprocess, os, glob, cv2
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
def sh(c): subprocess.run(c, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for s in SCENES:
    name=s['name']; d=f'{DEMOS}/{name}'; os.makedirs(d, exist_ok=True)
    src=f'/content/{name}_src'
    print('下載', name)
    subprocess.run(['curl','-sL','-A','Mozilla/5.0','-o',src,s['url']], check=True)
    before=f'{d}/before.mp4'
    sh(['ffmpeg','-y','-i',src,'-t',str(s['sec']),'-an','-r',str(FPS),'-vf',f'scale=-2:{H}','-c:v','libx264','-pix_fmt','yuv420p',before])
    # YOLO 逐幀標籤 → 寫 after（先 mp4v 再轉 H264）
    raw=f'/content/{name}_after_raw.mp4'; vw=None
    for r in model.track(before, tracker='bytetrack.yaml', conf=CONF, stream=True, verbose=False):
        ann=r.plot()
        if vw is None:
            hh,ww=ann.shape[:2]; vw=cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (ww,hh))
        vw.write(ann)
    vw.release()
    after=f'{d}/after.mp4'
    sh(['ffmpeg','-y','-i',raw,'-c:v','libx264','-crf','26','-pix_fmt','yuv420p','-movflags','+faststart',after])
    print('  完成', name, '->', d)
print('全部完成')

下載 pedestrians
  完成 pedestrians -> /content/drive/MyDrive/yolo-course/demos/pedestrians
下載 street_lagos
  完成 street_lagos -> /content/drive/MyDrive/yolo-course/demos/street_lagos
下載 leeds_1888
  完成 leeds_1888 -> /content/drive/MyDrive/yolo-course/demos/leeds_1888
全部完成


In [12]:
# 5b) 分割示範（yolo11n-seg）：對 street_lagos 產生 before/after（像素級遮罩）
import os, cv2, subprocess, shutil
from ultralytics import YOLO
def sh(c): subprocess.run(c, check=True)
seg = YOLO('yolo11n-seg.pt')
sname = 'street_lagos'
sbefore = f'{DEMOS}/{sname}/before.mp4'   # 沿用偵測階段裁好的原片
sd = f'{DEMOS}/seg_{sname}'; os.makedirs(sd, exist_ok=True)
raw = '/content/seg_after_raw.mp4'; vw = None
for r in seg.predict(sbefore, conf=0.35, stream=True, verbose=False):
    ann = r.plot()
    if vw is None:
        hh, ww = ann.shape[:2]
        vw = cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*'mp4v'), 15, (ww, hh))
    vw.write(ann)
vw.release()
shutil.copy(sbefore, f'{sd}/before.mp4')
sh(['ffmpeg','-y','-i',raw,'-c:v','libx264','-crf','26','-pix_fmt','yuv420p','-movflags','+faststart',f'{sd}/after.mp4'])
print('分割示範完成 ->', sd)

分割示範完成 -> /content/drive/MyDrive/yolo-course/demos/seg_street_lagos


In [13]:
# 5) flush 回 Drive（會卸載；最後一格）
from google.colab import drive
drive.flush_and_unmount()
print('已 flush，demos/ 各場景 before.mp4 + after.mp4 已寫回 Drive')

已 flush，demos/ 各場景 before.mp4 + after.mp4 已寫回 Drive
